# Arbeitspaket (AP) 1: Management & Nutzung Relationale Daten
# Kernel 3.12.11

## Python Libraries und Settings

In [1]:
# Libraries
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings("ignore")

# Working directory
print(os.getcwd())

/workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/AP01 Mock Exam


## <b>Vorbereitung (Dieser Teil wird <u>nicht</u> bewertet!)</b>

#### <b>1.) Starten Sie eine GitHub Codespaces Instanz auf Basis Ihres Forks des folgenden GitHub Repositories:</b>

##### GitHub-Repository: https://github.com/mario-gellrich-zhaw/python_postgresql_pgadmin


#### <b>2.) Erstellen und Testen Sie eine Datenbankverbindung mit der 'postgres' Datenbank.</b>

In [2]:
# Set up für Datenbankverbindung
user = "pgadmin"
password = "geheim"
host = "localhost"
port = "5432"
database = "postgres"

# Erstellen der Connection URL
db_connection_url = "postgresql://" + user + ":" + password +\
                    "@" + host + ":" + port + "/" + database

# Erstellen SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Test der Datenbankverbindung
with engine.connect() as connection:
    result = connection.execute(text('SELECT current_database()'))
    print(result.fetchone())

# Verbindung trennen
engine.dispose()

('postgres',)


#### <b>3.) Lesen Sie die Tabellenblätter aus 'bike_shop_data.xlsx' ein, und speichern Sie diese in separaten DataFrames.</b>

In [3]:
# Pfad zur .xlsx Datei
path = 'bike_shop_data.xlsx'

# Einlesen der Daten aus Excel File
df_bicycles = pd.read_excel(path, sheet_name='Bicycles')
df_customers = pd.read_excel(path, sheet_name='Customers')
df_sales = pd.read_excel(path, sheet_name='Sales')
df_salespersons = pd.read_excel(path, sheet_name='SalesPersons')
df_mechanics = pd.read_excel(path, sheet_name='Mechanics')
df_servicerecords = pd.read_excel(path, sheet_name='ServiceRecords')

# Anpassen der Spaltennamen zu 'lowercase'
df_bicycles.columns = df_bicycles.columns.str.lower()
df_customers.columns = df_customers.columns.str.lower()
df_sales.columns = df_sales.columns.str.lower()
df_salespersons.columns = df_salespersons.columns.str.lower()
df_mechanics.columns = df_mechanics.columns.str.lower()
df_servicerecords.columns = df_servicerecords.columns.str.lower()


#### <b>4.) Schreiben Sie die erstellten DataFrames in separate Tabellen in Ihrer Datenbank.</b>

In [4]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# Schreiben der DataFrames in die Datenbank (ohne zusätzliche Index-Spalte)
df_bicycles.to_sql('bicycles', engine, if_exists='replace', index=False)
df_customers.to_sql('customers', engine, if_exists='replace', index=False)
df_sales.to_sql('sales', engine, if_exists='replace', index=False)
df_salespersons.to_sql('salespersons', engine, if_exists='replace', index=False)
df_mechanics.to_sql('mechanics', engine, if_exists='replace', index=False)
df_servicerecords.to_sql('servicerecords', engine, if_exists='replace', index=False)

# Datenbankverbindung trennen
engine.dispose()

<div>

#### <b>Hinweis!</b>

#### <b>Falls etwas mit der PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für die folgenden Aufgaben die SQLite-Datenbank 'bike_shop.db'.</b>

#### <b>Erstellen Sie dazu die Engine in den Aufgaben-Zellen mit `create_engine('sqlite:///bike_shop.db')` statt mit `create_engine(db_connection_url)`.</b>

Das folgende Python-Codebeispiel demonstriert den Verbindungsaufbau mit der SQLite-Datenbank:

```python
# Engine für Datenbankverbindung erstellen
engine = create_engine('sqlite:///bike_shop.db')

# SQL Abfrage
df_sub = pd.read_sql_query("""SELECT *
                              FROM bicycles;""",
                          con=engine)

# Schliessen der Datenbankverbindung
engine.dispose()

df_sub
```

<b>Unterschiede SQLite gegenüber PostgreSQL (nur relevant, falls Sie SQLite verwenden):</b>
<ul>
  <li>Jahr aus einem Datum: <code>CAST(strftime('%Y', ServiceDate) AS INTEGER)</code> statt <code>EXTRACT(YEAR FROM ServiceDate)</code>.</li>
  <li>Datumswerte sind in SQLite als Text mit Uhrzeit gespeichert (z.B. '2023-12-31 00:00:00.000000'). Verwenden Sie für Datumsbereiche <code>DATE(SaleDate) BETWEEN '2023-03-01' AND '2023-12-31'</code>, sonst fehlt der letzte Tag.</li>
  <li>Runden: <code>ROUND(AVG(Price), 2)</code> funktioniert direkt, ein <code>CAST(... AS NUMERIC)</code> ist nicht nötig (aber auch nicht falsch).</li>
</ul>
</div>

## <b>Aufgaben (Dieser Teil wird bewertet!)</b>
<b>Hinweise zu den folgenden Aufgabenstellungen:</b>
<ul>
  <li>Erstellen Sie je Aufgabe eine SQL-Abfrage und integrieren Sie diese in Ihren Python Code.</li>
  <li>Speichern Sie je Aufgabe die Ergebnistabelle in einem DataFrame und stellen Sie diesen ganz oder in Teilen (z.B. erste 5 Zeilen) dar.</li>
  <li>Die Gross-/Kleinschreibung der Spaltennamen in der Ergebnistabelle wird nicht bewertet. PostgreSQL wandelt Spaltennamen und Aliase ohne Anführungszeichen automatisch in Kleinbuchstaben um (z.B. <code>BicycleCount</code> → <code>bicyclecount</code>).</li>
  <li>Runden in PostgreSQL: <code>ROUND()</code> mit Nachkommastellen funktioniert nur mit dem Datentyp NUMERIC, z.B. <code>ROUND(CAST(AVG(Price) AS NUMERIC), 2)</code>.</li>
  <li>Prozentzeichen in SQL-Abfragen (z.B. bei <code>LIKE</code>) müssen im Python-Code verdoppelt werden: <code>LIKE 'Gear%%'</code> statt <code>LIKE 'Gear%'</code>, sonst bricht die Abfrage mit einem Fehler ab.</li>
</ul>
<b style="color:red;">Beachten Sie, dass für die Punktevergabe auch die weiteren Anforderungen zu den Fragen unter 'Details zur Aufgabenstellung' herangezogen werden.</b>

#### <b>Aufgabe (1): Bestimmen Sie die Anzahl Bicycles pro Status.</b>
<b>Details zur Aufgabenstellung:</b>  
- Benötigte Tabelle: Bicycles  
- Benötigte Spalten: Status  
- Die Ergebnistabelle soll die Spalten enthalten: Status, BicycleCount.  
- Sortieren Sie die Ausgabe nach BicycleCount (absteigend).  

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [5]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_sub = pd.read_sql_query("""
    SELECT
        Status,
        COUNT(*) AS BicycleCount
    FROM Bicycles
    GROUP BY Status
    ORDER BY BicycleCount DESC;
    """,
    con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_sub

,status,bicyclecount
0,Sold,684
1,Available,672
2,In Maintenance,644


#### <b>Aufgabe (2): Listen Sie den durchschnittlichen Listenpreis (Spalte Price) der Bicycles für jede Kombination aus Marke und Modell auf.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabelle: Bicycles.
- Benötigte Spalten: Price, Make, Model.
- Die Ergebnistabelle soll die Spalten Make, Model und AveragePrice enthalten.
- Runden Sie den Durchschnittspreis auf zwei Nachkommastellen.
- Sortieren Sie die Ausgabe absteigend nach Make und absteigend nach AveragePrice.

<b>Tipp:</b>
- Eine Sortierung über zwei Spalten kann z.B. mittels 'ORDER BY Spalte A DESC, Spalte B DESC' erfolgen.

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [6]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_sub = pd.read_sql_query("""
    SELECT
        Make,
        Model,
        ROUND(CAST(AVG(Price) AS NUMERIC), 2) AS AveragePrice
    FROM Bicycles
    GROUP BY Make, Model
    ORDER BY Make DESC, AveragePrice DESC;
    """,
    con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_sub

,make,model,averageprice
0,Trek,Marlin,4666.03
1,Trek,Domane,4466.53
2,Trek,Emonda,4119.89
3,Specialized,Rockhopper,4742.53
4,Specialized,Roubaix,4504.83
5,Specialized,Allez,4128.30
6,Scott,Aspect,4297.69
7,Scott,Addict,4178.61
8,Scott,Spark,4091.72
9,Santa Cruz,Bronson,4747.68


#### <b>Aufgabe (3): Ermitteln Sie, wie viele Bicycles pro Marke einen Preis von 2'000.- bis 4'000.- USD haben.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabelle: Bicycles.
- Benötigte Spalten: Make, Price.
- Die Ergebnistabelle soll die Spalten Make und NumberOfBikes enthalten.
- Sortieren Sie die Ausgabe absteigend nach NumberOfBikes.

<b>Tipp:</b>
- Ein Werte- oder Datumsbereich kann mittels 'BETWEEN' und 'AND' definiert werden.
- Dabei ist 'BETWEEN' inklusive der verwendeten Angaben an beiden Enden.
- z.B. WHERE Value BETWEEN 22 AND 45 schliesst 22 und 45 mit ein.
- z.B. WHERE Datum BETWEEN '2021-01-01' AND '2021-12-31' beinhaltet in PostgreSQL das gesamte Jahr 2021 (sofern die Datumswerte keine Uhrzeit enthalten, was in diesen Daten der Fall ist).

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [7]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_sub = pd.read_sql_query("""
    SELECT
        Make,
        COUNT(*) AS NumberOfBikes
    FROM Bicycles
    WHERE Price BETWEEN 2000 AND 4000
    GROUP BY Make
    ORDER BY NumberOfBikes DESC;
    """,
    con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_sub

,make,numberofbikes
0,Cannondale,66
1,Giant,64
2,Specialized,57
3,Bianchi,57
4,Santa Cruz,54
5,Cervelo,54
6,Cube,52
7,Merida,50
8,Scott,45
9,Trek,42


#### <b>Aufgabe (4): Ermitteln Sie die durchschnittlichen Kosten für den Service 'Gear shifting adjustment' pro Jahr.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabelle: ServiceRecords
- Benötigte Spalten: ServiceDate, ServiceDescription, Cost
- Die Ergebnistabelle soll die Spalten Year und AverageServiceCosts enthalten.
- Runden Sie AverageServiceCosts auf zwei Nachkommastellen.
- Sortieren Sie die Ausgabe absteigend nach AverageServiceCosts.

<b>Tipp:</b>
- Mit CAST(EXTRACT(YEAR FROM ServiceDate) AS INTEGER) AS Year können Sie das Jahr als Ganzzahl aus ServiceDate extrahieren.

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [8]:
# SQLite-Variante: CAST(strftime('%Y', ServiceDate) AS INTEGER) statt CAST(EXTRACT(YEAR FROM ServiceDate) AS INTEGER)

# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_sub = pd.read_sql_query("""
    SELECT
        CAST(EXTRACT(YEAR FROM ServiceDate) AS INTEGER) AS Year,
        ROUND(CAST(AVG(Cost) AS NUMERIC), 2) AS AverageServiceCosts
    FROM ServiceRecords
    WHERE ServiceDescription = 'Gear shifting adjustment'
    GROUP BY CAST(EXTRACT(YEAR FROM ServiceDate) AS INTEGER)
    ORDER BY AverageServiceCosts DESC;
    """,
    con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_sub

,year,averageservicecosts
0,2019,25.83
1,2021,25.71
2,2020,25.40
3,2022,25.30
4,2023,25.07
5,2024,24.59


#### <b>Aufgabe (5): Ermitteln Sie, wie viele Bicycles jeder Verkäufer verkauft hat und zu welchem Gesamtpreis.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabellen: Salespersons, Sales
- Benötigte Spalten:
  - aus Salespersons: SalespersonID, FirstName, LastName, Phone, Email
  - aus Sales: SaleID, SalePrice, SalespersonID
- Die Ergebnistabelle soll die Spalten FirstName, LastName, Phone, Email, TotalBicyclesSold, TotalSalesAmount enthalten.
- Sortieren Sie die Ausgabe absteigend nach TotalSalesAmount.

<b>Tipp:</b>
- In PostgreSQL ist ein JOIN standardmässig ein INNER JOIN.

<b style="color: gray;">(max. erreichbare Punkte: 8)</b>

In [9]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_sub = pd.read_sql_query("""
    SELECT
        sp.FirstName,
        sp.LastName,
        sp.Phone,
        sp.Email,
        COUNT(s.SaleID) AS TotalBicyclesSold,
        SUM(s.SalePrice) AS TotalSalesAmount
    FROM Salespersons AS sp
    INNER JOIN Sales AS s ON sp.SalespersonID = s.SalespersonID
    GROUP BY sp.SalespersonID, sp.FirstName, sp.LastName, sp.Phone, sp.Email
    ORDER BY TotalSalesAmount DESC;
    """,
    con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_sub

,firstname,lastname,phone,email,totalbicyclessold,totalsalesamount
0,Heather,Brown,+41 (0)46 288 75 19,heather.brown@example.com,82,267840.0
1,Christine,Harris,+41 (0)61 523 86 58,christine.harris@example.com,74,237420.0
2,Karen,Randall,+41 (0)66 616 17 40,karen.randall@example.com,75,232950.0
3,Robert,Lyons,+41 (0)40 422 23 33,robert.lyons@example.com,69,232330.0
4,Lauren,Robinson,+41 (0)71 214 12 13,lauren.robinson@example.com,69,216840.0
5,Ryan,Baxter,+41 (0)40 706 53 70,ryan.baxter@example.com,72,208490.0
6,Nicole,Carter,+41 (0)77 266 76 74,nicole.carter@example.com,66,207940.0
7,Cathy,Becker,+41 (0)23 263 98 44,cathy.becker@example.com,67,200340.0
8,Michael,Zhang,+41 (0)63 311 51 92,michael.zhang@example.com,59,180090.0
9,Brian,Klein,+41 (0)77 449 26 62,brian.klein@example.com,51,174610.0


#### <b>Aufgabe (6): Ermitteln Sie, welche Bicycles der Marke Bianchi von März bis Dezember 2023 verkauft wurden, zu welchem Verkaufspreis, an welche Kunden und durch welche Verkäufer.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabellen: Bicycles, Sales, Customers, Salespersons
- Benötigte Spalten:
  - aus Bicycles: BicycleID, Make, Model
  - aus Sales: BicycleID, CustomerID, SalespersonID, SalePrice, SaleDate
  - aus Customers: CustomerID, FirstName, LastName
  - aus Salespersons: SalespersonID, FirstName, LastName
- Die Ergebnistabelle soll die Spalten BicycleID, Make, Model, SalePrice, SaleDate, CustomerFirstName, CustomerLastName, SalespersonFirstName, SalespersonLastName enthalten.
- Sortieren Sie die Ausgabe aufsteigend nach SaleDate.
- Beschränken Sie die Ausgabe auf 10 Zeilen.

<b style="color: gray;">(max. erreichbare Punkte: 8)</b>

In [10]:
# SQLite-Variante: DATE(s.SaleDate) BETWEEN '2023-03-01' AND '2023-12-31'

# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_sub = pd.read_sql_query("""
    SELECT
        b.BicycleID,
        b.Make,
        b.Model,
        s.SalePrice,
        s.SaleDate,
        cu.FirstName AS CustomerFirstName,
        cu.LastName AS CustomerLastName,
        sp.FirstName AS SalespersonFirstName,
        sp.LastName AS SalespersonLastName
    FROM Sales AS s
    INNER JOIN Bicycles AS b ON s.BicycleID = b.BicycleID
    INNER JOIN Customers AS cu ON s.CustomerID = cu.CustomerID
    INNER JOIN Salespersons AS sp ON s.SalespersonID = sp.SalespersonID
    WHERE b.Make = 'Bianchi'
      AND s.SaleDate BETWEEN '2023-03-01' AND '2023-12-31'
    ORDER BY s.SaleDate ASC
    LIMIT 10;
    """,
    con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_sub

,bicycleid,make,model,saleprice,saledate,customerfirstname,customerlastname,salespersonfirstname,salespersonlastname
0,C28CVDTTSB,Bianchi,Oltre,1680,2023-04-27,Rebecca,Henderson,Christine,Harris
1,CINFLGUWPI,Bianchi,Aria,2220,2023-05-24,Jacqueline,Waters,Cathy,Becker
2,C0VVD5LONX,Bianchi,Oltre,2360,2023-06-06,Jesus,Johnston,Christine,Harris
3,C9FCLAZ5BS,Bianchi,Aria,3020,2023-06-16,Emma,Stewart,Michael,Zhang
4,CVSWF385E4,Bianchi,Oltre,2460,2023-07-31,Jonathan,Camacho,Nicole,Carter
5,CJ21FVDXKP,Bianchi,Infinito,1700,2023-08-31,Amy,Gomez,Michael,Zhang
6,C4Y0NPNTOK,Bianchi,Aria,4660,2023-09-16,Anna,Phillips,Heather,Brown
7,CXYG3TWNY0,Bianchi,Aria,2070,2023-09-30,Denise,Jones,Christine,Harris
8,CZ2M1IWTJO,Bianchi,Oltre,4860,2023-10-14,Christina,Morgan,Brian,Klein
9,C1U99AVROS,Bianchi,Oltre,3120,2023-10-24,Patrick,Lopez,Robert,Lyons


#### <b>Aufgabe (7): Ermitteln Sie über alle Jahre hinweg (nicht pro Jahr), wie hoch die durchschnittlichen Servicekosten pro Mechaniker und Fahrradmarke waren.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabellen: Mechanics, ServiceRecords, Bicycles
- Benötigte Spalten:
  - aus Mechanics: MechanicID, FirstName, LastName
  - aus ServiceRecords: MechanicID, BicycleID, Cost, ServiceDate
  - aus Bicycles: BicycleID, Make
- Die Ergebnistabelle soll die Spalten MechanicFirstName, MechanicLastName, Make, ServicesCount, AverageServiceCosts, LastServiceDate enthalten.
- Runden Sie AverageServiceCosts auf zwei Nachkommastellen.
- Sortieren Sie die Ausgabe absteigend nach AverageServiceCosts.

<b>Tipp:</b>
- LastServiceDate kann mit Hilfe von MAX(sr.ServiceDate) AS LastServiceDate ermittelt werden (sr = Tabelle ServiceRecords).

<b style="color: gray;">(max. erreichbare Punkte: 8)</b>

In [11]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_sub = pd.read_sql_query("""
    SELECT
        m.FirstName AS MechanicFirstName,
        m.LastName AS MechanicLastName,
        b.Make,
        COUNT(*) AS ServicesCount,
        ROUND(CAST(AVG(sr.Cost) AS NUMERIC), 2) AS AverageServiceCosts,
        MAX(sr.ServiceDate) AS LastServiceDate
    FROM Mechanics AS m
    INNER JOIN ServiceRecords AS sr ON m.MechanicID = sr.MechanicID
    INNER JOIN Bicycles AS b ON sr.BicycleID = b.BicycleID
    GROUP BY m.MechanicID, m.FirstName, m.LastName, b.Make
    ORDER BY AverageServiceCosts DESC;
    """,
    con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_sub

,mechanicfirstname,mechaniclastname,make,servicescount,averageservicecosts,lastservicedate
0,Shannon,Larsen,Scott,65,39.76,2024-08-07
1,Cynthia,Pittman,Cannondale,74,38.61,2024-07-28
2,Shannon,Larsen,Specialized,75,37.41,2024-08-20
3,Courtney,Ford,Santa Cruz,82,36.41,2024-08-04
4,Erin,Williamson,Specialized,69,36.14,2024-08-17
...,...,...,...,...,...,...
95,Cynthia,Pittman,Specialized,82,27.53,2024-08-19
96,Lisa,Hall,Bianchi,70,27.47,2024-08-26
97,Cynthia,Pittman,Bianchi,77,26.81,2024-07-20
98,Melissa,Miller,Bianchi,71,26.49,2024-08-26


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [12]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')


-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-21 20:49:48
Python Version: 3.12.11
IP Address: 127.0.0.1
-----------------------------------
